In [5]:
import os, sys, subprocess

REPO_URL = "https://github.com/madihakomal75/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if "google.colab" in sys.modules:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.exists("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Verify grain and window bounds
id_col = "url_hash_id" if "url_hash_id" in df.columns else ("page_id" if "page_id" in df.columns else df.columns[0])
total_rows = len(df)
unique_ids = df[id_col].nunique()

print(f"Dataset Grain Verification:")
print(f"  Total Rows: {total_rows:,}")
print(f"  Unique Page IDs ({id_col}): {unique_ids:,}")
print(f"  Is grain 1 row per unique page? {total_rows == unique_ids}")
print(f"  Observed Time Window Features: 'content_age_days' (max: {df['content_age_days'].max()}), 'days_since_last_update' (max: {df['days_since_last_update'].max()})")

Dataset Grain Verification:
  Total Rows: 30,000
  Unique Page IDs (content_id): 30,000
  Is grain 1 row per unique page? True
  Observed Time Window Features: 'content_age_days' (max: 564), 'days_since_last_update' (max: 373)


# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/madihakomal75/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*Unit of Analysis:
One row = One unique content page (identified by url_hash_id / content_hash_id).*

*Time Window:
Features are calculated over a 90-day observation window (impressions_90d, clicks_90d) ending at a fixed historical slice point (report_date). Temporal metadata features include content_age_days and days_since_last_update.*

## 2. Fields: feature / label / context / excluded

1. Target Label:
*  trend_direction / is_declining_label: Binary outcome target indicating performance decline (trend_direction == 'down').
2. Model Features:
*  content_age_days: Total age of the page in days.
*  days_since_last_update: Days elapsed since the last content update.
*  impressions_90d: Total GSC impressions over the last 90 days.
*  clicks_90d: Total GSC clicks over the last 90 days.
*  avg_position: Average search engine ranking position.
*  ctr: Click-through rate ($\text{clicks} / \text{impressions}$).
*  word_count: Total word count of the page body.
3. Context / Metadata Fields:
*  url_hash_id / client_hash_id: Anonymized primary keys for grouping, evaluation splits, and auditability.
*  content_type: Category or format of the content.
4. Excluded Fields (and Why):
*  search_volume: Excluded from raw feature importance due to near-zero correlation ($r \approx 0.001$) with actual observed page impressions.
*  Leakage features derived after the evaluation cutoff date.*

In [6]:
# Verify field buckets in dataset
feature_cols = ['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'word_count']
target_col = 'trend_direction'

print("Feature summary statistics:")
df[feature_cols].describe().T[['mean', 'std', 'min', '50%', 'max']]

Feature summary statistics:


,mean,std,min,50%,max
content_age_days,256.167800,132.707930,90.0,236.00,564.0
days_since_last_update,46.098300,42.078709,1.0,20.00,373.0
impressions_90d,5200.366300,16838.019547,1.0,731.00,517715.0
avg_position,16.342380,15.216790,0.0,10.80,245.0
ctr,0.510733,3.279162,0.0,0.07,100.0
word_count,3107.760325,1452.382598,8.0,2877.00,9546.0


## 3. Verify it with queries (grain, counts, missing values, windows)

Contract Verification Claims:
*  Zero Uniqueness Violations: Every row maps to a distinct page ID.
*  Missing Value Handling: Core numeric features have complete coverage or deterministic zero-fills.
*  Target Imbalance: Target class distribution is balanced enough for evaluation without extreme resampling.

In [7]:
duplicate_counts = df[id_col].duplicated().sum()

missing_summary = df[feature_cols].isnull().sum()

label_dist = df['trend_direction'].value_counts(normalize=True) * 100

print(f"1. Duplicate Keys Found: {duplicate_counts}")
print("\n2. Missing Values Per Feature:")
print(missing_summary.to_string())
print("\n3. Label Percentage Distribution:")
print(label_dist.to_string())

1. Duplicate Keys Found: 0

2. Missing Values Per Feature:
content_age_days             0
days_since_last_update       0
impressions_90d              0
avg_position                 0
ctr                          0
word_count                7699

3. Label Percentage Distribution:
trend_direction
down      54.206667
stable    19.873333
up        14.626667
new        7.453333
flat       3.840000


## 4. Data limits

*Data Limitations & Boundaries:
Search Console Only: Data reflects Google Search Console aggregate metrics and does not contain user on-page behavioral metrics (e.g., bounce rate, session duration).
Unbalanced Historical Depth: Pages created recently (content_age_days < 90) have truncated historical windows compared to older evergreen content.
Non-Causal Signals: High correlation between staleness (days_since_last_update) and rank decay does not guarantee that updating text will immediately restore search position.*

In [8]:

new_pages = (df['content_age_days'] < 90).sum()
stale_pages = (df['days_since_last_update'] > 365).sum()

print(f"Pages with < 90 days history: {new_pages:,} ({new_pages/len(df):.1%})")
print(f"Pages with > 365 days since update: {stale_pages:,} ({stale_pages/len(df):.1%})")

Pages with < 90 days history: 0 (0.0%)
Pages with > 365 days since update: 5 (0.0%)
